In [ ]:
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
import numpy as np


def load_mask(mask_path: str | Path) -> torch.Tensor:
    """
    Carrega uma máscara de segmentação como tensor LongTensor [H, W].
    Espera máscara com valores inteiros representando classes.
    """
    mask = Image.open(mask_path)
    mask = np.array(mask)
    
    # Caso venha RGB, pega apenas um canal.
    # Idealmente, sua máscara já deve estar salva como classe por pixel.
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    
    return torch.from_numpy(mask).long()


def compute_iou(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    num_classes: int,
    ignore_index: int | None = None,
) -> dict:
    """
    Calcula IoU por classe e mean IoU para uma imagem.
    """
    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"Shape diferente: ground truth={y_true.shape}, predito={y_pred.shape}"
        )

    if ignore_index is not None:
        valid_mask = y_true != ignore_index
        y_true = y_true[valid_mask]
        y_pred = y_pred[valid_mask]

    iou_per_class = {}

    for class_id in range(num_classes):
        true_class = y_true == class_id
        pred_class = y_pred == class_id

        intersection = torch.logical_and(true_class, pred_class).sum().item()
        union = torch.logical_or(true_class, pred_class).sum().item()

        if union == 0:
            iou = float("nan")
        else:
            iou = intersection / union

        iou_per_class[f"iou_class_{class_id}"] = iou

    valid_ious = [
        value for value in iou_per_class.values()
        if not np.isnan(value)
    ]

    iou_per_class["mean_iou"] = (
        float(np.mean(valid_ious)) if valid_ious else float("nan")
    )

    return iou_per_class


def evaluate_masks_to_csv(
    samples: list[dict],
    num_classes: int,
    output_csv: str | Path,
    ignore_index: int | None = None,
) -> pd.DataFrame:
    """
    Avalia IoU para uma lista de máscaras ground truth x predita.

    Parâmetros
    ----------
    samples:
        Lista no formato:
        [
            {
                "image_id": "img_001",
                "gt_path": "masks_gt/img_001.png",
                "pred_path": "masks_pred/img_001.png",
            },
            ...
        ]

    num_classes:
        Número total de classes da segmentação.

    output_csv:
        Caminho onde o CSV será salvo.

    ignore_index:
        Classe ignorada no cálculo, exemplo: 255.
    """
    results = []

    for sample in samples:
        image_id = sample["image_id"]
        gt_path = sample["gt_path"]
        pred_path = sample["pred_path"]

        y_true = load_mask(gt_path)
        y_pred = load_mask(pred_path)

        metrics = compute_iou(
            y_true=y_true,
            y_pred=y_pred,
            num_classes=num_classes,
            ignore_index=ignore_index,
        )

        row = {
            "image_id": image_id,
            "gt_path": str(gt_path),
            "pred_path": str(pred_path),
            **metrics,
        }

        results.append(row)

    df_metrics = pd.DataFrame(results)
    df_metrics.to_csv(output_csv, index=False)

    return df_metrics

In [12]:
from pathlib import Path
import csv

import torch
from PIL import Image
import numpy as np


def load_mask(mask_path: Path) -> torch.Tensor:
    """
    Carrega uma máscara de segmentação como tensor.
    """

    mask = Image.open(mask_path)
    mask = np.array(mask)

    if mask.ndim == 3:
        mask = mask[:, :, 0]

    return torch.from_numpy(mask).long()


def load_test_ids(test_txt_path: str) -> list[str]:
    """
    Lê os IDs presentes no arquivo test_sample.txt.

    Exemplo esperado do txt:
        image_001
        image_002
        image_003

    OU:
        image_001.png
        image_002.png
    """

    with open(test_txt_path, "r", encoding="utf-8") as f:
        ids = [line.strip() for line in f if line.strip()]

    return ids


def compute_iou(
    gt_mask: torch.Tensor,
    pred_mask: torch.Tensor,
    num_classes: int,
    ignore_index: int | None = None,
) -> dict:

    if gt_mask.shape != pred_mask.shape:
        raise ValueError(
            f"Shapes diferentes: GT {gt_mask.shape} vs Pred {pred_mask.shape}"
        )

    if ignore_index is not None:
        valid_mask = gt_mask != ignore_index
        gt_mask = gt_mask[valid_mask]
        pred_mask = pred_mask[valid_mask]

    iou_per_class = {}

    for cls in range(num_classes):

        gt_cls = gt_mask == cls
        pred_cls = pred_mask == cls

        intersection = torch.logical_and(gt_cls, pred_cls).sum().item()
        union = torch.logical_or(gt_cls, pred_cls).sum().item()

        if union == 0:
            iou = None
        else:
            iou = intersection / union

        iou_per_class[f"iou_class_{cls}"] = iou

    valid_ious = [
        v for v in iou_per_class.values()
        if v is not None
    ]

    mean_iou = (
        sum(valid_ious) / len(valid_ious)
        if valid_ious
        else None
    )

    return {
        **iou_per_class,
        "mean_iou": mean_iou,
    }


def evaluate_masks_to_csv(
    gt_dir: str,
    pred_dir: str,
    test_txt_path: str,
    output_csv: str,
    num_classes: int,
    ignore_index: int | None = None,
    extensions: tuple[str, ...] = (".png", ".tif", ".tiff"),
):
    """
    Avalia apenas as imagens listadas no test_sample.txt.
    """

    gt_dir = Path(gt_dir)
    pred_dir = Path(pred_dir)
    output_csv = Path(output_csv)

    image_ids = load_test_ids(test_txt_path)

    results = []

    for image_id in image_ids:

        image_id = image_id.strip()

        # Caso o txt venha sem extensão
        if not Path(image_id).suffix:

            found = False

            for ext in extensions:
                possible_name = f"{image_id}{ext}"

                gt_path = gt_dir / possible_name
                pred_path = pred_dir / possible_name

                if gt_path.exists() and pred_path.exists():
                    image_filename = possible_name
                    found = True
                    break

            if not found:
                print(f"[AVISO] Arquivo não encontrado: {image_id}")
                continue

        else:
            image_filename = image_id

            gt_path = gt_dir / image_filename
            pred_path = pred_dir / image_filename

            if not gt_path.exists():
                print(f"[AVISO] GT não encontrado: {gt_path}")
                continue

            if not pred_path.exists():
                print(f"[AVISO] Predição não encontrada: {pred_path}")
                continue

        gt_mask = load_mask(gt_path)
        pred_mask = load_mask(pred_path)

        metrics = compute_iou(
            gt_mask=gt_mask,
            pred_mask=pred_mask,
            num_classes=num_classes,
            ignore_index=ignore_index,
        )

        results.append({
            "image_id": image_filename,
            "gt_path": str(gt_path),
            "pred_path": str(pred_path),
            **metrics,
        })

        print(
            f"[OK] {image_filename} | "
            f"Mean IoU: {metrics['mean_iou']:.4f}"
        )

    if not results:
        raise RuntimeError("Nenhuma imagem foi avaliada.")

    fieldnames = list(results[0].keys())

    output_csv.parent.mkdir(parents=True, exist_ok=True)

    with output_csv.open("w", newline="", encoding="utf-8") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()
        writer.writerows(results)

    print(f"\nCSV salvo em: {output_csv}")
    print(f"Total de imagens avaliadas: {len(results)}")


def filter_iou_csv(
    input_csv: str,
    output_csv: str,
    min_iou: float,
    max_iou: float,
    metric_col: str = "mean_iou",
):

    input_csv = Path(input_csv)
    output_csv = Path(output_csv)

    filtered_rows = []

    with input_csv.open("r", encoding="utf-8") as f:

        reader = csv.DictReader(f)

        for row in reader:

            value = row.get(metric_col)

            if value in (None, "", "None"):
                continue

            iou = float(value)

            if min_iou <= iou <= max_iou:
                filtered_rows.append(row)

    if not filtered_rows:
        print("Nenhuma imagem encontrada nessa faixa de IoU.")
        return

    output_csv.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with output_csv.open("w", newline="", encoding="utf-8") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=filtered_rows[0].keys(),
        )

        writer.writeheader()
        writer.writerows(filtered_rows)

    print(f"CSV filtrado salvo em: {output_csv}")
    print(f"Total de imagens filtradas: {len(filtered_rows)}")

In [13]:
predictions_dir = "/home/calebe/geo-projects-car/experiments/deeplabv3-resnet50-256_20260508_003335/test_eval/predictions/"
masks_dir = "/home/calebe/data/masks_replicated_full"
satellites_dir = "/home/calebe/data/satelite_images"
test_list = "/home/calebe/geo-projects-car/data-segments/test_sample.txt"
experiment_plot_path = "/home/calebe/geo-projects-car/experiments/deeplabv3-resnet50-256_20260508_003335/test_eval/plotting_test_images/"

In [14]:
evaluate_masks_to_csv(
    gt_dir=masks_dir,
    pred_dir=predictions_dir,
    test_txt_path=test_list,
    output_csv="/home/calebe/geo-projects-car/experiments/deeplabv3-resnet50-256_20260508_003335/test_eval/experiments_iou_by_image.csv",
    num_classes=5,
    ignore_index=255,
)

[OK] ES-3203502-82E3092110494428857501E7CFE14E7F.tif | Mean IoU: 0.3612
[OK] ES-3204500-9CC4B82933F14C55A54830546C7329CD.tif | Mean IoU: 0.3530
[OK] ES-3201803-7541B0853F1F493AA1AF1CB5FFBE86B6.tif | Mean IoU: 0.3866
[OK] ES-3204005-1B50467AB5ED4177861696409CA3DF29.tif | Mean IoU: 0.3888
[OK] ES-3204104-1D4D03D2089B458189D0EC938C77A47D.tif | Mean IoU: 0.3661
[OK] ES-3201704-D3D8D1212EEC4248B741CEAAF3D55719.tif | Mean IoU: 0.5169
[OK] ES-3202207-0E6C4C3E2C7C4E75ABBE4CA2A06DF1A4.tif | Mean IoU: 0.4713
[OK] ES-3202504-A17A58AA50184BA1A2BAC61A8BCF4620.tif | Mean IoU: 0.4781
[OK] ES-3203346-10EBEF09FC3F4E7483BF05C846063EBA.tif | Mean IoU: 0.6709
[OK] ES-3203163-67784E45AE8D43739469DDC4DF6E77C7.tif | Mean IoU: 0.9215
[OK] ES-3203304-59DD091CEDBC4EAC907C505FDED01BB1.tif | Mean IoU: 0.5097
[OK] ES-3204906-A2C88D541B0943358C9BC3EB40A6B6D6.tif | Mean IoU: 0.5524
[OK] ES-3201506-CDEB18EC56ED4780BBB1034848CF48A6.tif | Mean IoU: 0.3782
[OK] ES-3201902-607B9B19565948AEADF6734ACF1B9B3C.tif | Mean IoU: